# Generate MultiSocial Vietnamese Dataset (Colab, NVIDIA NIM)

This notebook creates `multisocial_micro_vi.csv` from your own uploaded CSV file using NVIDIA NIM (OpenAI-compatible).

## Required packages

- openai
- pandas
- tqdm
- scikit-learn

## Expected input

Upload a CSV using this schema:
`["Unnamed: 0", "Emotion", "Sentence"]`

## Before running

Set your NVIDIA API key in Colab Secrets or environment variables:

- `NVIDIA_API_KEY`

## Execution

This notebook processes all sampled rows with NVIDIA NIM (`openai/gpt-oss-20b`).
The model is a reasoning model, so generation uses streaming and reads final `delta.content` while keeping a speed-optimized configuration for paraphrase tasks.


In [ ]:
# Install dependencies (Colab)
!pip -q install openai pandas tqdm scikit-learn ollama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import logging
import os
import time
from pathlib import Path
from typing import List, Tuple

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from openai import OpenAI

try:
    from google.colab import userdata
except Exception:
    userdata = None

# -----------------------------
# Global constants - NVIDIA NIM Configuration
# -----------------------------
RANDOM_SEED = 42
HUMAN_SAMPLE_SIZE = 2000
MODEL_NAME = 'openai/gpt-oss-20b'
OUTPUT_CSV_PATH = '/content/drive/MyDrive/multisocial_outputs/multisocial_micro_vi_v2.csv'
INPUT_CSV_PATH_DEFAULT = '/content/drive/MyDrive/multisocial_outputs/vsmec_df.csv'
MAX_RETRIES = 3
SUCCESS_CALL_SLEEP_SECONDS = 0.5
MAX_OUTPUT_TOKENS = 4096
GEN_TEMPERATURE = 0.1
GEN_TOP_P = 1.0

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
SOURCE_NAME = 'vsmec'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s'
)
logger = logging.getLogger('multisocial_vi')

nvidia_api_key = None
if userdata is not None:
    try:
        nvidia_api_key = userdata.get('NVIDIA_API_KEY')
    except Exception:
        nvidia_api_key = None

if not nvidia_api_key:
    nvidia_api_key = os.getenv('NVIDIA_API_KEY')

# Initialize the OpenAI Client pointing to NVIDIA NIM
client = OpenAI(
  base_url=NVIDIA_BASE_URL,
  api_key=nvidia_api_key
)

logger.info('NVIDIA NIM Client initialized.')
logger.info('Base URL: %s', NVIDIA_BASE_URL)
logger.info('Model: %s', MODEL_NAME)

In [ ]:
# Cell disabled to prevent overwriting Ollama configuration
print('Using configuration from previous cell.')
print(f'Active model: {MODEL_NAME}')

Using configuration from previous cell.
Active model: openai/gpt-oss-20b


In [ ]:
def paraphrase_vietnamese(text: str, client, model, max_retries: int = 3) -> tuple[bool, str]:
    """Generate one Vietnamese paraphrase using the NVIDIA NIM (OpenAI-compatible)."""
    prompt = (
        'Paraphrase the following Vietnamese social media post. Keep the exact same informal tone, '
        'length, hashtags, and emojis. IMPORTANT: Output the paraphrased text ONLY in Vietnamese. '
        'Do not translate it to English or any other language. Do not add conversational fillers. '
        'Just output the paraphrased text. Text: '
        f'{text}'
    )

    messages = [{'role': 'user', 'content': prompt}]

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=GEN_TEMPERATURE,
                top_p=GEN_TOP_P,
                max_tokens=MAX_OUTPUT_TOKENS,
                stream=False
            )

            paraphrased_text = completion.choices[0].message.content.strip()
            if paraphrased_text:
                return True, paraphrased_text
            else:
                logger.warning('Empty response content at attempt %s/%s', attempt, max_retries)

        except Exception as exc:
            logger.warning(
                'Paraphrasing failed at attempt %s/%s: %s',
                attempt,
                max_retries,
                exc,
            )

        if attempt < max_retries:
            sleep_seconds = 2 ** attempt
            logger.info('Sleeping %s second(s) before retry...', sleep_seconds)
            time.sleep(sleep_seconds)

    return False, ''

def load_human_data(csv_path: str = INPUT_CSV_PATH_DEFAULT, n_samples: int = 2000, seed: int = 42) -> pd.DataFrame:
    """Load and sample human-written Vietnamese social texts from CSV."""
    csv_file = Path(csv_path)
    if not csv_file.exists():
        raise FileNotFoundError(f'Input CSV not found: {csv_path}')

    raw_df = pd.read_csv(csv_file)
    if 'Sentence' not in raw_df.columns:
        raise ValueError("Input CSV must contain a 'Sentence' column.")

    text_series = raw_df['Sentence'].astype('string').fillna('').str.strip()
    text_series = text_series[text_series != '']

    if len(text_series) < n_samples:
        n_samples = len(text_series)

    sampled_texts = text_series.sample(n=n_samples, random_state=seed, replace=False).tolist()

    human_df = pd.DataFrame({
        'text': pd.Series(sampled_texts, dtype='string').fillna(''),
        'label': 0,
        'multi_label': 'human',
        'source': SOURCE_NAME,
    })
    return human_df

def generate_machine_data(human_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate machine-written counterparts."""
    successful_humans = []
    generated_rows = []
    for text in tqdm(human_df['text'].tolist(), desc='Generating (NVIDIA NIM)', unit='post'):
        success, paraphrased_text = paraphrase_vietnamese(text, client, MODEL_NAME)
        if success:
            successful_humans.append({'text': text, 'label': 0, 'multi_label': 'human', 'source': SOURCE_NAME})
            generated_rows.append({'text': paraphrased_text, 'label': 1, 'multi_label': MODEL_NAME, 'source': SOURCE_NAME})
        time.sleep(SUCCESS_CALL_SLEEP_SECONDS)

    return pd.DataFrame(successful_humans), pd.DataFrame(generated_rows)

def format_and_save(human_df: pd.DataFrame, machine_df: pd.DataFrame, output_path: str) -> pd.DataFrame:
    """Finalize dataset."""
    combined_df = pd.concat([human_df, machine_df], ignore_index=True)
    combined_df['text'] = combined_df['text'].astype(str)
    combined_df['language'] = 'vi'
    combined_df['length'] = combined_df['text'].apply(lambda x: len(x.split()))
    combined_df['potential_noise'] = 0

    combined_df['split'] = 'train'
    test_size = int(len(combined_df) * 0.2)
    test_indices = combined_df.sample(n=test_size, random_state=RANDOM_SEED).index
    combined_df.loc[test_indices, 'split'] = 'test'

    final_df = combined_df[['text', 'label', 'multi_label', 'split', 'language', 'length', 'source', 'potential_noise']]
    final_df.to_csv(output_path, index=False)
    return final_df

def main(input_csv_path: str) -> pd.DataFrame:
    human_df = load_human_data(input_csv_path, HUMAN_SAMPLE_SIZE, RANDOM_SEED)
    paired_h, paired_m = generate_machine_data(human_df)
    return format_and_save(paired_h, paired_m, OUTPUT_CSV_PATH)

In [ ]:
import os

# Run NVIDIA NIM pipeline on sampled Vietnamese rows
input_csv_path = INPUT_CSV_PATH_DEFAULT
if not os.path.exists(input_csv_path):
    raise FileNotFoundError(f"Could not find the file at {input_csv_path}. Please check the path.")

print(f"Using input CSV: {input_csv_path}")
print(f"Target Model: {MODEL_NAME}")
print(f"NVIDIA Base URL: {NVIDIA_BASE_URL}")

# Execute the main pipeline defined in fa01000c
final_df = main(input_csv_path=input_csv_path)

print('\nCompleted NVIDIA NIM vi run')
display(final_df.head())

Using input CSV: /content/drive/MyDrive/multisocial_outputs/vsmec_df.csv
Target Model: openai/gpt-oss-20b
NVIDIA Base URL: https://integrate.api.nvidia.com/v1


Generating (NVIDIA NIM): 100%|██████████| 2000/2000 [3:08:08<00:00,  5.64s/post]


Completed NVIDIA NIM vi run


,text,label,multi_label,split,language,length,source,potential_noise
0,chỉ dân đen là khổ thôi . chả trách được họ kh...,0,human,train,vi,22,vsmec,0
1,thôi xong . chia buồn cùng quý quốc,0,human,train,vi,8,vsmec,0
2,per tao bị con mặt nồi này dí 2 lần táp vào ch...,0,human,train,vi,33,vsmec,0
3,thanh xuân rồi cũng sẽ qua có những người tron...,0,human,train,vi,17,vsmec,0
4,một giây rung động con tim :))))),0,human,train,vi,7,vsmec,0


In [ ]:
# Quick verification checks (dynamic row count after failed generations are dropped)
expected_columns = [
    'text',
    'label',
    'multi_label',
    'split',
    'language',
    'length',
    'source',
    'potential_noise',
]
assert list(final_df.columns) == expected_columns, 'Column order does not match required schema.'

label_counts = final_df['label'].value_counts().to_dict()
assert set(label_counts.keys()) == {0, 1}, f'Unexpected labels found: {label_counts}'
assert label_counts[0] == label_counts[1], f'Class imbalance detected: {label_counts}'

split_counts = final_df.groupby(['split', 'label']).size().unstack(fill_value=0)
display(split_counts)
display(final_df['split'].value_counts())

assert (final_df['language'] == 'vi').all(), 'language must be vi for all rows.'
assert (final_df['source'] == SOURCE_NAME).all(), f'source must be {SOURCE_NAME} for all rows.'
assert (final_df['potential_noise'] == 0).all(), 'potential_noise must be 0 for all rows.'
assert not final_df.isna().any().any(), 'No missing values are allowed in final_df.'

print(f'Final paired rows per class: {label_counts[0]}')
print(f'Total rows: {len(final_df)}')

label,0,1
split,,
test,416,381
train,1577,1612


,count
split,
train,3189
test,797


Final paired rows per class: 1993
Total rows: 3986


In [ ]:
import shutil

drive_output_dir = Path('/content/drive/MyDrive/multisocial_outputs')
drive_output_dir.mkdir(parents=True, exist_ok=True)
drive_output_path = drive_output_dir / OUTPUT_CSV_PATH

if 'final_df' in globals():
    final_df.to_csv(drive_output_path, index=False)
elif Path(OUTPUT_CSV_PATH).exists():
    shutil.copy2(OUTPUT_CSV_PATH, drive_output_path)
else:
    raise FileNotFoundError(
        f'Could not find {OUTPUT_CSV_PATH}. Run the execution cell first to generate the dataset.'
    )

print(f'Saved output to: {drive_output_path}')
print(f'File size: {drive_output_path.stat().st_size / 1024:.2f} KB')

Saved output to: /content/drive/MyDrive/multisocial_outputs/multisocial_micro_vi_v2.csv
File size: 424.22 KB
